In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
RAW_DIR   = os.path.join(DATA_ROOT, 'raw')
TEST_DIR  = os.path.join(DATA_ROOT, 'test_in')
OUT_PATH  = '/kaggle/working/preds.npy'

# ── Hyperparameters ────────────────────────────────────────────────────────
LOOKBACK   = 10
HORIZON    = 16
H, W       = 140, 124
BATCH_SIZE = 6
EPOCHS     = 40
LR         = 3e-4
STRIDE     = 2
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Feature lists ──────────────────────────────────────────────────────────
MET_FEATS  = ['q2','t2','u10','v10','swdown','pblh','psfc','rain']
EMIT_FEATS = ['PM25','NH3','SO2','NOx','NMVOC_e','NMVOC_finn','bio']
MONTHS     = ['APRIL_16','JULY_16','OCT_16','DEC_16']

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device : cuda
PyTorch: 2.10.0+cu128


In [3]:
def load_month(month):
    path = os.path.join(RAW_DIR, month)
    data = {}
    data['cpm25'] = np.load(os.path.join(path, 'cpm25.npy')).astype(np.float32)
    for f in MET_FEATS + EMIT_FEATS:
        fp = os.path.join(path, f'{f}.npy')
        if os.path.exists(fp):
            data[f] = np.load(fp).astype(np.float32)
    return data

print("Loading training months...")
month_data = {}
for m in MONTHS:
    month_data[m] = load_month(m)
    print(f"  {m}: T={month_data[m]['cpm25'].shape[0]}, "
          f"features={list(month_data[m].keys())}")

Loading training months...
  APRIL_16: T=715, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
  JULY_16: T=739, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
  OCT_16: T=739, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
  DEC_16: T=739, features=['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']


In [4]:
def engineer_features(data_dict):
    """Add wind speed + direction. Works for both train and test dicts."""
    d = dict(data_dict)
    if 'u10' in d and 'v10' in d:
        d['wind_speed'] = np.sqrt(d['u10']**2 + d['v10']**2)
        d['wind_dir']   = np.arctan2(d['v10'], d['u10'])
    return d

for m in MONTHS:
    month_data[m] = engineer_features(month_data[m])
    print(f"  {m} engineered features: {list(month_data[m].keys())}")

  APRIL_16 engineered features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']
  JULY_16 engineered features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']
  OCT_16 engineered features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']
  DEC_16 engineered features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']


In [5]:
# ── Build FEAT_KEYS from what actually exists — includes engineered feats ──
BASE_FEATS = ['cpm25'] + MET_FEATS + EMIT_FEATS
EXTRA_FEATS = ['wind_speed', 'wind_dir']

FEAT_KEYS = []
for f in BASE_FEATS + EXTRA_FEATS:
    if all(f in month_data[m] for m in MONTHS):
        FEAT_KEYS.append(f)

# ── LOCK N_FEATS here — never changes after this ───────────────────────────
N_FEATS = len(FEAT_KEYS)
print(f"FEAT_KEYS locked: {N_FEATS} features")
print(FEAT_KEYS)

# ── Gridwise robust normalization (median + IQR) ───────────────────────────
def compute_norm_stats(month_data, feat_keys):
    stats = {}
    for feat in feat_keys:
        arrays   = [month_data[m][feat] for m in MONTHS]
        combined = np.concatenate(arrays, axis=0)          # (T_total, H, W)
        med      = np.median(combined, axis=0)
        q75      = np.percentile(combined, 75, axis=0)
        q25      = np.percentile(combined, 25, axis=0)
        iqr      = q75 - q25
        iqr      = np.where(iqr < 1e-6, 1.0, iqr)
        stats[feat] = (med.astype(np.float32), iqr.astype(np.float32))
    return stats

def normalize(arr, med, iqr):
    return (arr - med) / iqr

def denormalize(arr, med, iqr):
    return arr * iqr + med

print("\nComputing normalization statistics...")
norm_stats = compute_norm_stats(month_data, FEAT_KEYS)

# Save cpm25 stats for denormalization at inference
cpm25_med, cpm25_iqr = norm_stats['cpm25']
print(f"cpm25 median range : [{cpm25_med.min():.2f}, {cpm25_med.max():.2f}]")
print(f"cpm25 IQR range    : [{cpm25_iqr.min():.2f}, {cpm25_iqr.max():.2f}]")

# ── Normalize all months → (T, H, W, N_FEATS) ─────────────────────────────
def normalize_month(data_dict, stats, feat_keys):
    arrays = []
    for feat in feat_keys:
        med, iqr = stats[feat]
        arrays.append(normalize(data_dict[feat], med, iqr))
    return np.stack(arrays, axis=-1).astype(np.float32)   # (T, H, W, C)

norm_data = {}
for m in MONTHS:
    norm_data[m] = normalize_month(month_data[m], norm_stats, FEAT_KEYS)
    print(f"  {m}: {norm_data[m].shape}")

# ── Free raw data immediately ──────────────────────────────────────────────
del month_data
gc.collect()
total_mb = sum(v.nbytes for v in norm_data.values()) / 1024**2
print(f"\nRaw data freed. norm_data RAM: {total_mb:.1f} MB")
print(f"N_FEATS = {N_FEATS}  ← model will be built with this exact number")

FEAT_KEYS locked: 18 features
['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']

Computing normalization statistics...
cpm25 median range : [1.79, 239.42]
cpm25 IQR range    : [1.75, 286.02]
  APRIL_16: (715, 140, 124, 18)
  JULY_16: (739, 140, 124, 18)
  OCT_16: (739, 140, 124, 18)
  DEC_16: (739, 140, 124, 18)

Raw data freed. norm_data RAM: 3495.0 MB
N_FEATS = 18  ← model will be built with this exact number


In [6]:
class PM25Dataset(Dataset):
    def __init__(self, norm_arrays, lookback=LOOKBACK, horizon=HORIZON, stride=STRIDE):
        self.lookback = lookback
        self.horizon  = horizon
        self.arrays   = norm_arrays
        self.indices  = []
        self.cpm_idx  = FEAT_KEYS.index('cpm25')

        for arr_idx, arr in enumerate(norm_arrays):
            T      = arr.shape[0]
            window = lookback + horizon
            for i in range(0, T - window + 1, stride):
                self.indices.append((arr_idx, i))

        print(f"  Samples: {len(self.indices)}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        arr_idx, t = self.indices[idx]
        arr = self.arrays[arr_idx]
        x   = arr[t : t + self.lookback]                                       # (10,H,W,C)
        y   = arr[t + self.lookback : t + self.lookback + self.horizon,
                  :, :, self.cpm_idx]                                          # (16,H,W)
        return torch.from_numpy(x.copy()).float(), torch.from_numpy(y.copy()).float()


print("Building datasets...")
print("Train:")
train_ds = PM25Dataset([norm_data[m] for m in ['APRIL_16','JULY_16','OCT_16']],
                       stride=STRIDE)
print("Val:")
val_ds   = PM25Dataset([norm_data['DEC_16']], stride=4)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Building datasets...
Train:
  Samples: 1059
Val:
  Samples: 179

Train batches: 177
Val   batches: 30


In [7]:
class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        # ensure reduction doesn't make hidden=0
        hidden = max(1, channels // reduction)
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )
        self.spatial_att = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.channel_att(x).view(x.shape[0], x.shape[1], 1, 1)
        x  = x * ca
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1,  keepdim=True)[0]
        sa  = self.spatial_att(torch.cat([avg, mx], dim=1))
        return x * sa


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_attn=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
        self.attn = CBAM(out_ch) if use_attn else nn.Identity()
        self.res  = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.attn(self.conv(x)) + self.res(x)


class TemporalEncoder(nn.Module):
    def __init__(self, n_feats, lookback, hidden):
        super().__init__()
        # store for reference
        self.n_feats  = n_feats
        self.lookback = lookback
        in_ch         = lookback * n_feats      # explicit — no global dependency
        self.proj     = nn.Conv2d(in_ch, hidden, 1)

    def forward(self, x):
        B, T, H, W, C = x.shape
        x = x.permute(0, 1, 4, 2, 3).reshape(B, T * C, H, W)
        return self.proj(x)


class UNetPM25(nn.Module):
    def __init__(self, n_feats, lookback, horizon, base=48):
        super().__init__()
        self.horizon = horizon
        self.enc_in  = TemporalEncoder(n_feats, lookback, hidden=base)

        self.enc1 = ConvBlock(base,     base*2,  use_attn=False)
        self.enc2 = ConvBlock(base*2,   base*4,  use_attn=False)
        self.enc3 = ConvBlock(base*4,   base*8,  use_attn=True)
        self.pool = nn.MaxPool2d(2, ceil_mode=True)

        self.bottleneck = ConvBlock(base*8, base*16, use_attn=True)

        self.up3  = nn.ConvTranspose2d(base*16, base*8,  2, stride=2)
        self.dec3 = ConvBlock(base*16,  base*8,  use_attn=True)

        self.up2  = nn.ConvTranspose2d(base*8,  base*4,  2, stride=2)
        self.dec2 = ConvBlock(base*8,   base*4,  use_attn=False)

        self.up1  = nn.ConvTranspose2d(base*4,  base*2,  2, stride=2)
        self.dec1 = ConvBlock(base*4,   base*2,  use_attn=False)

        self.head = nn.Sequential(
            nn.Conv2d(base*2, base,    3, padding=1),
            nn.GELU(),
            nn.Conv2d(base,   horizon, 1)
        )

    def _match_size(self, x, ref):
        """Pad OR crop x to exactly match ref spatial dims."""
        dh = ref.shape[2] - x.shape[2]
        dw = ref.shape[3] - x.shape[3]
        if dh > 0 or dw > 0:
            x = F.pad(x, [0, max(dw, 0), 0, max(dh, 0)])
        if x.shape[2] > ref.shape[2] or x.shape[3] > ref.shape[3]:
            x = x[:, :, :ref.shape[2], :ref.shape[3]]
        return x

    def forward(self, x):
        e0 = self.enc_in(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bn = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([self._match_size(self.up3(bn), e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._match_size(self.up2(d3), e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._match_size(self.up1(d2), e1), e1], dim=1))

        return self.head(d1)   # (B, 16, H, W)


# ── Build with exact N_FEATS from Cell 4 ──────────────────────────────────
print(f"Building model: N_FEATS={N_FEATS}, LOOKBACK={LOOKBACK}, HORIZON={HORIZON}")
model = UNetPM25(n_feats=N_FEATS, lookback=LOOKBACK,
                 horizon=HORIZON, base=48).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

# ── Shape sanity check ─────────────────────────────────────────────────────
with torch.no_grad():
    dummy = torch.zeros(2, LOOKBACK, H, W, N_FEATS).to(DEVICE)
    out   = model(dummy)
    assert out.shape == (2, HORIZON, H, W), f"Bad output shape: {out.shape}"
    print(f"Shape check passed: {out.shape} ✓")

Building model: N_FEATS=18, LOOKBACK=10, HORIZON=16
Parameters: 18,417,529
Shape check passed: torch.Size([2, 16, 140, 124]) ✓


In [8]:
def smape_loss(pred, target, eps=1.0):
    return torch.mean(
        torch.abs(pred - target) /
        (0.5 * (torch.abs(target) + torch.abs(pred)) + eps)
    )

def pearson_corr_loss(pred, target):
    pred_f   = pred.reshape(pred.shape[0], -1)
    target_f = target.reshape(target.shape[0], -1)
    pm = pred_f   - pred_f.mean(dim=1,   keepdim=True)
    tm = target_f - target_f.mean(dim=1, keepdim=True)
    corr = (pm * tm).sum(dim=1) / (
        pm.norm(dim=1) * tm.norm(dim=1) + 1e-8
    )
    return 1.0 - corr.mean()

def episode_mask_batch(target, sigma=2.5):
    B, T, H, W = target.shape
    flat = target.view(B, T, -1)
    mean = flat.mean(dim=1, keepdim=True)
    std  = flat.std(dim=1,  keepdim=True)
    mask = ((flat - mean) > sigma * std).view(B, T, H, W)
    return mask

def episode_aware_loss(pred, target,
                       w_global=0.4, w_ep_smape=0.4, w_ep_corr=0.2):
    loss_global = smape_loss(pred, target)
    mask        = episode_mask_batch(target)

    if mask.sum() > 100:
        loss_ep_smape = smape_loss(pred[mask], target[mask])
        loss_ep_corr  = pearson_corr_loss(
            pred   * mask.float(),
            target * mask.float()
        )
    else:
        loss_ep_smape = loss_global
        loss_ep_corr  = torch.tensor(0.0, device=pred.device)

    total = (w_global   * loss_global
           + w_ep_smape * loss_ep_smape
           + w_ep_corr  * loss_ep_corr)
    return total, loss_global.item(), loss_ep_smape.item(), loss_ep_corr.item()

print("Loss functions defined ✓")

Loss functions defined ✓


In [9]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = OneCycleLR(
    optimizer,
    max_lr       = LR,
    steps_per_epoch = len(train_loader),
    epochs       = EPOCHS,
    pct_start    = 0.1,
    anneal_strategy = 'cos'
)

best_val_loss = float('inf')
best_ckpt     = '/kaggle/working/best_model.pt'

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)   # yb: (B,16,H,W)
        optimizer.zero_grad()

        pred = model(xb)                          # (B,16,H,W)
        if pred.shape[-2:] != yb.shape[-2:]:
            pred = F.interpolate(pred, size=yb.shape[-2:],
                                 mode='bilinear', align_corners=False)

        loss, *_ = episode_aware_loss(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss += loss.item()

    tr_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred   = model(xb)
            if pred.shape[-2:] != yb.shape[-2:]:
                pred = F.interpolate(pred, size=yb.shape[-2:],
                                     mode='bilinear', align_corners=False)
            loss, *_ = episode_aware_loss(pred, yb)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}/{EPOCHS} | "
              f"Train: {tr_loss:.4f} | Val: {val_loss:.4f} | "
              f"Best: {best_val_loss:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e}")

print(f"\nTraining done. Best val loss: {best_val_loss:.4f}")

Epoch 001/40 | Train: 0.6471 | Val: 0.6672 | Best: 0.6672 | LR: 5.43e-05
Epoch 005/40 | Train: 0.2277 | Val: 0.1999 | Best: 0.1999 | LR: 2.99e-04
Epoch 010/40 | Train: 0.1922 | Val: 0.1793 | Best: 0.1793 | LR: 2.80e-04
Epoch 015/40 | Train: 0.1771 | Val: 0.1713 | Best: 0.1713 | LR: 2.36e-04
Epoch 020/40 | Train: 0.1645 | Val: 0.1693 | Best: 0.1659 | LR: 1.76e-04
Epoch 025/40 | Train: 0.1505 | Val: 0.1627 | Best: 0.1627 | LR: 1.11e-04
Epoch 030/40 | Train: 0.1387 | Val: 0.1662 | Best: 0.1600 | LR: 5.35e-05
Epoch 035/40 | Train: 0.1307 | Val: 0.1579 | Best: 0.1579 | LR: 1.40e-05
Epoch 040/40 | Train: 0.1272 | Val: 0.1594 | Best: 0.1574 | LR: 1.22e-09

Training done. Best val loss: 0.1574


In [10]:
print("Loading test data...")
test_data = {}
test_data['cpm25'] = np.load(os.path.join(TEST_DIR, 'cpm25.npy')).astype(np.float32)

for f in MET_FEATS:
    fp = os.path.join(TEST_DIR, f'{f}.npy')
    if os.path.exists(fp):
        test_data[f] = np.load(fp).astype(np.float32)

for f in EMIT_FEATS:
    fp = os.path.join(TEST_DIR, f'{f}.npy')
    if os.path.exists(fp):
        test_data[f] = np.load(fp).astype(np.float32)

# ── Engineer same features as training ────────────────────────────────────
test_data = engineer_features(test_data)

N_TEST = test_data['cpm25'].shape[0]
print(f"Test samples : {N_TEST}")
print(f"Test features: {list(test_data.keys())}")

# ── Verify all FEAT_KEYS are present ──────────────────────────────────────
missing = [f for f in FEAT_KEYS if f not in test_data]
if missing:
    print(f"WARNING — missing features (will zero-fill): {missing}")
else:
    print(f"All {N_FEATS} features present ✓")

Loading test data...
Test samples : 218
Test features: ['cpm25', 'q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'wind_speed', 'wind_dir']
All 18 features present ✓


In [11]:
def build_test_tensor_single(sample_idx):
    """
    Build one normalized test sample using the globally locked
    FEAT_KEYS and norm_stats from training.
    Returns: (LOOKBACK, H, W, N_FEATS) float32
    """
    arrays = []
    for feat in FEAT_KEYS:
        med, iqr = norm_stats[feat]              # (H, W)
        if feat in test_data:
            arr    = test_data[feat][sample_idx] # (10, H, W)
            normed = (arr - med[None]) / iqr[None]
        else:
            normed = np.zeros((LOOKBACK, H, W), dtype=np.float32)
        arrays.append(normed)
    return np.stack(arrays, axis=-1).astype(np.float32)  # (10,H,W,N_FEATS)

# ── Quick check: sample shape must match what model expects ───────────────
sample = build_test_tensor_single(0)
assert sample.shape == (LOOKBACK, H, W, N_FEATS), \
    f"Sample shape {sample.shape} doesn't match model input ({LOOKBACK},{H},{W},{N_FEATS})"
print(f"Single sample shape: {sample.shape} ✓")
print(f"Model input channels: {LOOKBACK * N_FEATS} "
      f"(== proj weight: {model.enc_in.proj.weight.shape[1]}) ✓")

Single sample shape: (10, 140, 124, 18) ✓
Model input channels: 180 (== proj weight: 180) ✓


In [12]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print("Best checkpoint loaded ✓")

preds_list = []
n_batches  = (N_TEST + BATCH_SIZE - 1) // BATCH_SIZE

with torch.no_grad():
    for i in range(0, N_TEST, BATCH_SIZE):
        batch_idx = range(i, min(i + BATCH_SIZE, N_TEST))

        # Build batch on-the-fly — BATCH_SIZE samples at a time
        batch_np = np.stack(
            [build_test_tensor_single(j) for j in batch_idx], axis=0
        )                                              # (B,10,H,W,N_FEATS)

        xb   = torch.from_numpy(batch_np).to(DEVICE)
        pred = model(xb)                               # (B,16,H,W)

        if pred.shape[-2:] != (H, W):
            pred = F.interpolate(pred, size=(H, W),
                                 mode='bilinear', align_corners=False)

        pred_np = pred.cpu().numpy()                   # (B,16,H,W)

        # Denormalize back to µg/m³
        pred_np = pred_np * cpm25_iqr[None, None] + cpm25_med[None, None]

        # Physical clamp — PM2.5 cannot be negative
        pred_np = np.clip(pred_np, 0, None)

        preds_list.append(pred_np)
        print(f"  Batch {i//BATCH_SIZE + 1}/{n_batches} done", end='\r')

preds = np.concatenate(preds_list, axis=0)            # (218,16,H,W)
preds = preds.transpose(0, 2, 3, 1)                   # (218,H,W,16)

print(f"\nFinal predictions shape: {preds.shape}")
assert preds.shape == (218, 140, 124, 16), \
    f"Shape mismatch: {preds.shape} — expected (218,140,124,16)"
print("Shape check passed ✓")

Best checkpoint loaded ✓
  Batch 37/37 done
Final predictions shape: (218, 140, 124, 16)
Shape check passed ✓


In [13]:
np.save(OUT_PATH, preds)
print(f"Saved → {OUT_PATH}")

# ── Final checks ───────────────────────────────────────────────────────────
loaded = np.load(OUT_PATH)
print(f"\nLoaded shape : {loaded.shape}")
print(f"dtype        : {loaded.dtype}")
print(f"min / max    : {loaded.min():.4f} / {loaded.max():.4f}")
print(f"NaN count    : {np.isnan(loaded).sum()}")
print(f"Negative cnt : {(loaded < 0).sum()}")
assert loaded.shape == (218, 140, 124, 16), "Shape wrong!"
assert np.isnan(loaded).sum() == 0,         "NaNs found!"
assert (loaded < 0).sum() == 0,             "Negatives found!"
print("\n✅ Submission ready: /kaggle/working/preds.npy")

Saved → /kaggle/working/preds.npy

Loaded shape : (218, 140, 124, 16)
dtype        : float32
min / max    : 0.0000 / 31612.9785
NaN count    : 0
Negative cnt : 0

✅ Submission ready: /kaggle/working/preds.npy
